In [ ]:
!pip install -q tensorflow scikit-learn pandas tqdm opencv-python

In [ ]:
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras import Model, Input
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Concatenate
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger

In [ ]:
DATA_DIR = Path('/kaggle/input/asl-alphabet')
MODELS_DIR = Path('/kaggle/working/models')
RESULTS_DIR = Path('/kaggle/working/results')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
@dataclass
class DataConfig:
    data_dir: Path
    image_size: tuple = (200, 200)
    batch_size: int = 32
    val_split: float = 0.15
    test_split: float = 0.15
    seed: int = 42
    grayscale: bool = True


class ASLDataLoader:
    TRAIN_SUBDIR = 'asl_alphabet_train'

    def __init__(self, config):
        self.config = config
        self.train_dir = Path(config.data_dir) / self.TRAIN_SUBDIR
        self._df = None
        self.classes = None
        self.class_to_idx = None

    @property
    def num_classes(self):
        return len(self.classes) if self.classes else 0

    @property
    def df(self):
        if self._df is None:
            self._df = self._scan_dataset()
        return self._df

    def _scan_dataset(self):
        class_dirs = sorted(d for d in self.train_dir.iterdir() if d.is_dir())
        self.classes = [d.name for d in class_dirs]
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        records = [
            {'filepath': str(img), 'label': cls_dir.name, 'class_idx': self.class_to_idx[cls_dir.name]}
            for cls_dir in class_dirs
            for img in cls_dir.glob('*.jpg')
        ]
        return pd.DataFrame(records)

    def split(self):
        val_test_size = self.config.val_split + self.config.test_split
        relative_test = self.config.test_split / val_test_size
        train_df, temp_df = train_test_split(
            self.df, test_size=val_test_size, stratify=self.df['label'], random_state=self.config.seed,
        )
        val_df, test_df = train_test_split(
            temp_df, test_size=relative_test, stratify=temp_df['label'], random_state=self.config.seed,
        )
        return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

    def image_generator(self, df, augment=False):
        aug_args = {
            'rotation_range': 10,
            'width_shift_range': 0.1,
            'height_shift_range': 0.1,
            'zoom_range': 0.1,
            'horizontal_flip': False,
        } if augment else {}
        datagen = ImageDataGenerator(rescale=1.0 / 255, **aug_args)
        return datagen.flow_from_dataframe(
            dataframe=df,
            x_col='filepath',
            y_col='label',
            directory=None,
            target_size=self.config.image_size,
            color_mode='grayscale' if self.config.grayscale else 'rgb',
            batch_size=self.config.batch_size,
            class_mode='categorical',
            classes=self.classes,
            shuffle=augment,
            seed=self.config.seed,
        )

In [ ]:
@dataclass
class MobileNetConfig:
    input_shape: tuple = (200, 200, 1)
    num_classes: int = 29
    dropout_rate: float = 0.4
    l2_lambda: float = 1e-4
    learning_rate_head: float = 3e-4
    learning_rate_finetune: float = 1e-5
    unfreeze_from_layer: int = 100


def build_mobilenet(config):
    base = MobileNetV2(
        input_shape=(config.input_shape[0], config.input_shape[1], 3),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = False
    inputs = Input(shape=config.input_shape)
    x = Concatenate()([inputs, inputs, inputs])
    x = base(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(config.l2_lambda))(x)
    x = Dropout(config.dropout_rate)(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(config.l2_lambda))(x)
    x = Dropout(config.dropout_rate)(x)
    outputs = Dense(config.num_classes, activation='softmax')(x)
    model = Model(inputs, outputs, name='mobilenet_transfer')
    model.compile(optimizer=Adam(config.learning_rate_head), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def unfreeze_for_finetuning(model, config):
    base = model.get_layer('mobilenetv2_1.00_224')
    for layer in base.layers[:config.unfreeze_from_layer]:
        layer.trainable = False
    for layer in base.layers[config.unfreeze_from_layer:]:
        layer.trainable = True
    model.compile(optimizer=Adam(config.learning_rate_finetune), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
mobilenet_config = MobileNetConfig(input_shape=(200, 200, 1), num_classes=29)

loader = ASLDataLoader(DataConfig(data_dir=DATA_DIR, image_size=(200, 200), batch_size=64))
train_df, val_df, test_df = loader.split()

mobilenet_config.num_classes = loader.num_classes

train_gen = loader.image_generator(train_df, augment=True)
val_gen   = loader.image_generator(val_df,   augment=False)
test_gen  = loader.image_generator(test_df,  augment=False)

print(f'Classes: {loader.num_classes}')
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

In [ ]:
model = build_mobilenet(mobilenet_config)
model.summary()

In [ ]:
t0 = time.time()

callbacks_head = [
    EarlyStopping(monitor='val_accuracy', patience=5, min_delta=1e-4, restore_best_weights=True),
    ModelCheckpoint(str(MODELS_DIR / 'mobilenet_best.h5'), monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    CSVLogger(str(RESULTS_DIR / 'mobilenet_head_log.csv')),
]

history_head = model.fit(train_gen, validation_data=val_gen, epochs=15, callbacks=callbacks_head)
pd.DataFrame(history_head.history).to_csv(RESULTS_DIR / 'mobilenet_head_history.csv', index=False)

In [ ]:
unfreeze_for_finetuning(model, mobilenet_config)

callbacks_ft = [
    EarlyStopping(monitor='val_accuracy', patience=5, min_delta=1e-4, restore_best_weights=True),
    ModelCheckpoint(str(MODELS_DIR / 'mobilenet_best.h5'), monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    CSVLogger(str(RESULTS_DIR / 'mobilenet_finetune_log.csv')),
]

history_ft = model.fit(train_gen, validation_data=val_gen, epochs=10, callbacks=callbacks_ft)
train_time = time.time() - t0
print(f'Total train time: {train_time:.1f}s')

pd.DataFrame(history_ft.history).to_csv(RESULTS_DIR / 'mobilenet_finetune_history.csv', index=False)

In [ ]:
best = load_model(str(MODELS_DIR / 'mobilenet_best.h5'))
test_gen.reset()
probs = best.predict(test_gen, verbose=1)
y_pred = np.argmax(probs, axis=1)
y_true = test_gen.classes

print(f'Accuracy: {accuracy_score(y_true, y_pred):.4f}')
print(classification_report(y_true, y_pred, target_names=loader.classes))